# EnergyWise - Exploratory Data Analysis (01_eda)

Exploratory analysis of the joined hourly energy + weather table
(`data/processed/hourly_energy_weather.parquet`).

For each of the four buildings (academic, facilities, lecture, library) this
notebook shows:

1. `energy_kwh` over the full date range, with missing hours left as visible gaps (no filling or interpolation).
2. Average daily profile by hour of day, weekday and weekend as separate lines.
3. Average energy by day of week.
4. Monthly average energy (seasonality).
5. Energy against temperature (scatter + binned means).
6. Heatmap of mean energy by hour-of-day x day-of-week.
7. Histogram of energy_kwh.
8. Calendar-style heatmap of days with missing hours.

Each figure is followed by a "My observations" cell to fill in manually.
Only pandas and matplotlib are used.


In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.rcParams["figure.figsize"] = (13, 4)
plt.rcParams["axes.grid"] = True

_DATA_CANDIDATES = [
    Path("data/processed/hourly_energy_weather.parquet"),
    Path("../data/processed/hourly_energy_weather.parquet"),
    Path.cwd() / "data/processed/hourly_energy_weather.parquet",
    Path.cwd().parent / "data/processed/hourly_energy_weather.parquet",
]
DATA_PATH = next((p for p in _DATA_CANDIDATES if p.exists()), None)
if DATA_PATH is None:
    raise FileNotFoundError("Could not find data/processed/hourly_energy_weather.parquet")

print("Loading:", DATA_PATH)
df = pd.read_parquet(DATA_PATH)

df["timestamp_ist"] = pd.to_datetime(df["timestamp_ist"])
if df["timestamp_ist"].dt.tz is None:
    df["timestamp_ist"] = df["timestamp_ist"].dt.tz_localize("Asia/Kolkata")

BUILDINGS = sorted(df["building"].unique().tolist())
DOW_LABELS = ["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"]
MONTH_LABELS = ["Jan", "Feb", "Mar", "Apr", "May", "Jun",
                "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"]

print("Rows:", len(df), "| Buildings:", BUILDINGS)
print("Range:", df["timestamp_ist"].min(), "->", df["timestamp_ist"].max())


In [ ]:
def building_frame(name):
    return df[df["building"] == name].sort_values("timestamp_ist").reset_index(drop=True)


def valid_rows(bdf):
    return bdf[bdf["energy_kwh"].notna()].copy()


def fig_energy_timeseries(name):
    bdf = building_frame(name)
    fig, ax = plt.subplots()
    # NaNs are kept, so missing hours appear as gaps in the line.
    ax.plot(bdf["timestamp_ist"], bdf["energy_kwh"], linewidth=0.6, color="tab:blue")
    ax.set_title(name + ": hourly energy_kwh over time (missing hours = gaps)")
    ax.set_xlabel("timestamp (IST)")
    ax.set_ylabel("energy_kwh")
    fig.tight_layout()
    plt.show()


def fig_daily_profile(name):
    bdf = valid_rows(building_frame(name))
    bdf["hour"] = bdf["timestamp_ist"].dt.hour
    bdf["is_weekend"] = bdf["timestamp_ist"].dt.dayofweek >= 5
    prof = bdf.groupby(["hour", "is_weekend"])["energy_kwh"].mean().unstack("is_weekend")
    fig, ax = plt.subplots()
    ax.plot(prof.index, prof[False], marker="o", label="Weekday (Mon-Fri)")
    ax.plot(prof.index, prof[True], marker="o", label="Weekend (Sat-Sun)")
    ax.set_title(name + ": average daily profile by hour of day")
    ax.set_xlabel("hour of day (IST)")
    ax.set_ylabel("mean energy_kwh")
    ax.set_xticks(range(0, 24))
    ax.legend()
    fig.tight_layout()
    plt.show()


def fig_day_of_week(name):
    bdf = valid_rows(building_frame(name))
    bdf["dow"] = bdf["timestamp_ist"].dt.dayofweek
    dow = bdf.groupby("dow")["energy_kwh"].mean()
    fig, ax = plt.subplots()
    ax.bar([DOW_LABELS[i] for i in dow.index], dow.values, color="tab:green")
    ax.set_title(name + ": average energy by day of week")
    ax.set_xlabel("day of week")
    ax.set_ylabel("mean energy_kwh")
    fig.tight_layout()
    plt.show()


def fig_monthly(name):
    bdf = valid_rows(building_frame(name))
    bdf["month"] = bdf["timestamp_ist"].dt.month
    mon = bdf.groupby("month")["energy_kwh"].mean()
    fig, ax = plt.subplots()
    ax.bar([MONTH_LABELS[m - 1] for m in mon.index], mon.values, color="tab:orange")
    ax.set_title(name + ": monthly average energy (seasonality)")
    ax.set_xlabel("month")
    ax.set_ylabel("mean energy_kwh")
    fig.tight_layout()
    plt.show()


def fig_vs_temperature(name):
    bdf = building_frame(name)
    sub = bdf[bdf["energy_kwh"].notna() & bdf["temperature_c"].notna()]
    fig, ax = plt.subplots()
    ax.scatter(sub["temperature_c"], sub["energy_kwh"], s=4, alpha=0.08,
               color="tab:blue", label="hourly observations")
    bins = np.linspace(sub["temperature_c"].min(), sub["temperature_c"].max(), 21)
    cat = pd.cut(sub["temperature_c"], bins)
    bm = sub.groupby(cat, observed=True)["energy_kwh"].mean()
    centers = [iv.mid for iv in bm.index]
    ax.plot(centers, bm.values, color="red", marker="o", label="binned mean")
    ax.set_title(name + ": energy vs temperature")
    ax.set_xlabel("temperature (deg C)")
    ax.set_ylabel("energy_kwh")
    ax.legend()
    fig.tight_layout()
    plt.show()


def fig_heatmap_hour_dow(name):
    bdf = valid_rows(building_frame(name))
    bdf["hour"] = bdf["timestamp_ist"].dt.hour
    bdf["dow"] = bdf["timestamp_ist"].dt.dayofweek
    pivot = bdf.pivot_table(index="dow", columns="hour", values="energy_kwh", aggfunc="mean")
    fig, ax = plt.subplots()
    im = ax.imshow(pivot.values, aspect="auto", origin="upper", cmap="viridis")
    ax.set_xticks(range(len(pivot.columns)))
    ax.set_xticklabels(pivot.columns)
    ax.set_yticks(range(len(pivot.index)))
    ax.set_yticklabels([DOW_LABELS[i] for i in pivot.index])
    ax.set_title(name + ": mean energy by hour-of-day x day-of-week")
    ax.set_xlabel("hour of day")
    ax.set_ylabel("day of week")
    fig.colorbar(im, ax=ax, label="mean energy_kwh")
    fig.tight_layout()
    plt.show()


def fig_histogram(name):
    bdf = valid_rows(building_frame(name))
    fig, ax = plt.subplots()
    ax.hist(bdf["energy_kwh"], bins=60, color="tab:purple")
    ax.set_title(name + ": distribution of hourly energy_kwh")
    ax.set_xlabel("energy_kwh")
    ax.set_ylabel("count of hours")
    fig.tight_layout()
    plt.show()


def fig_missing_calendar(name):
    bdf = building_frame(name).copy()
    bdf["date"] = bdf["timestamp_ist"].dt.date
    daily = bdf.groupby("date")["is_missing"].sum()
    start = min(daily.index)
    end = max(daily.index)
    n_days = (end - start).days + 1
    weeks = (n_days // 7) + 2
    grid = np.full((7, weeks), np.nan)
    for day, value in daily.items():
        grid[day.weekday(), (day - start).days // 7] = value
    fig, ax = plt.subplots(figsize=(14, 3))
    im = ax.imshow(grid, aspect="auto", origin="upper", cmap="magma_r", vmin=0, vmax=24)
    ax.set_yticks(range(7))
    ax.set_yticklabels(DOW_LABELS)
    ticks, labels = [], []
    for year in range(start.year, end.year + 1):
        d = max(start, pd.Timestamp(year, 1, 1).date())
        if d <= end:
            ticks.append((d - start).days // 7)
            labels.append(str(year))
    ax.set_xticks(ticks)
    ax.set_xticklabels(labels)
    ax.set_title(name + ": calendar heatmap of missing hours per day")
    fig.colorbar(im, ax=ax, label="missing hours in day")
    fig.tight_layout()
    plt.show()


# Building: `academic`


### 1. energy_kwh over the full date range (missing hours = gaps)


In [ ]:
fig_energy_timeseries("academic")


## My observations

TODO: write 2-3 sentences


### 2. Average daily profile by hour of day (weekday vs weekend)


In [ ]:
fig_daily_profile("academic")


## My observations

TODO: write 2-3 sentences


### 3. Average energy by day of week


In [ ]:
fig_day_of_week("academic")


## My observations

TODO: write 2-3 sentences


### 4. Monthly average energy (seasonality)


In [ ]:
fig_monthly("academic")


## My observations

TODO: write 2-3 sentences


### 5. Energy vs temperature


In [ ]:
fig_vs_temperature("academic")


## My observations

TODO: write 2-3 sentences


### 6. Mean energy by hour-of-day x day-of-week


In [ ]:
fig_heatmap_hour_dow("academic")


## My observations

TODO: write 2-3 sentences


### 7. Histogram of energy_kwh


In [ ]:
fig_histogram("academic")


## My observations

TODO: write 2-3 sentences


### 8. Calendar heatmap of days with missing hours


In [ ]:
fig_missing_calendar("academic")


## My observations

TODO: write 2-3 sentences


# Building: `facilities`


### 1. energy_kwh over the full date range (missing hours = gaps)


In [ ]:
fig_energy_timeseries("facilities")


## My observations

TODO: write 2-3 sentences


### 2. Average daily profile by hour of day (weekday vs weekend)


In [ ]:
fig_daily_profile("facilities")


## My observations

TODO: write 2-3 sentences


### 3. Average energy by day of week


In [ ]:
fig_day_of_week("facilities")


## My observations

TODO: write 2-3 sentences


### 4. Monthly average energy (seasonality)


In [ ]:
fig_monthly("facilities")


## My observations

TODO: write 2-3 sentences


### 5. Energy vs temperature


In [ ]:
fig_vs_temperature("facilities")


## My observations

TODO: write 2-3 sentences


### 6. Mean energy by hour-of-day x day-of-week


In [ ]:
fig_heatmap_hour_dow("facilities")


## My observations

TODO: write 2-3 sentences


### 7. Histogram of energy_kwh


In [ ]:
fig_histogram("facilities")


## My observations

TODO: write 2-3 sentences


### 8. Calendar heatmap of days with missing hours


In [ ]:
fig_missing_calendar("facilities")


## My observations

TODO: write 2-3 sentences


# Building: `lecture`


### 1. energy_kwh over the full date range (missing hours = gaps)


In [ ]:
fig_energy_timeseries("lecture")


## My observations

TODO: write 2-3 sentences


### 2. Average daily profile by hour of day (weekday vs weekend)


In [ ]:
fig_daily_profile("lecture")


## My observations

TODO: write 2-3 sentences


### 3. Average energy by day of week


In [ ]:
fig_day_of_week("lecture")


## My observations

TODO: write 2-3 sentences


### 4. Monthly average energy (seasonality)


In [ ]:
fig_monthly("lecture")


## My observations

TODO: write 2-3 sentences


### 5. Energy vs temperature


In [ ]:
fig_vs_temperature("lecture")


## My observations

TODO: write 2-3 sentences


### 6. Mean energy by hour-of-day x day-of-week


In [ ]:
fig_heatmap_hour_dow("lecture")


## My observations

TODO: write 2-3 sentences


### 7. Histogram of energy_kwh


In [ ]:
fig_histogram("lecture")


## My observations

TODO: write 2-3 sentences


### 8. Calendar heatmap of days with missing hours


In [ ]:
fig_missing_calendar("lecture")


## My observations

TODO: write 2-3 sentences


# Building: `library`


### 1. energy_kwh over the full date range (missing hours = gaps)


In [ ]:
fig_energy_timeseries("library")


## My observations

TODO: write 2-3 sentences


### 2. Average daily profile by hour of day (weekday vs weekend)


In [ ]:
fig_daily_profile("library")


## My observations

TODO: write 2-3 sentences


### 3. Average energy by day of week


In [ ]:
fig_day_of_week("library")


## My observations

TODO: write 2-3 sentences


### 4. Monthly average energy (seasonality)


In [ ]:
fig_monthly("library")


## My observations

TODO: write 2-3 sentences


### 5. Energy vs temperature


In [ ]:
fig_vs_temperature("library")


## My observations

TODO: write 2-3 sentences


### 6. Mean energy by hour-of-day x day-of-week


In [ ]:
fig_heatmap_hour_dow("library")


## My observations

TODO: write 2-3 sentences


### 7. Histogram of energy_kwh


In [ ]:
fig_histogram("library")


## My observations

TODO: write 2-3 sentences


### 8. Calendar heatmap of days with missing hours


In [ ]:
fig_missing_calendar("library")


## My observations

TODO: write 2-3 sentences


# Summary table per building


In [ ]:
summary = df.groupby("building").agg(
    mean_energy_kwh=("energy_kwh", "mean"),
    min_energy_kwh=("energy_kwh", "min"),
    max_energy_kwh=("energy_kwh", "max"),
    missing_pct=("is_missing", lambda s: s.mean() * 100.0),
).round(4)
print(summary)
summary
